# Manuscript R-free and R-work tables — Matthew + Vipul

Recreate the experimental Mpro tables at the end of `multi_xray_ms/main.tex`
(Tables 3–6), adding Vipul's **1-state / 2-condition** and **2-state / 4-condition** results.

For each `(states, conditions, target dataset)`, select the **lowest R-free** across
all available condition combinations that include that dataset. Report **R-work
from the same selected model**, never a separate R-work minimum. Different table
cells may therefore refer to different models and different condition combinations.
Values are fractions, displayed to three decimals.

**Data note:** `vipul_1s2c_phenix_ref/summary.csv` contains model paths under
`out/vipul_1s2c/` without `_phenix_ref`. Its refinement status is unverified.
Those values are included **as supplied** in the Phenix-summary tables, marked †.
No values are inferred or recomputed from structures. The source configuration below
can be updated when a corrected summary is available.

Run all cells with Python, pandas, and IPython. No IMP, CCTBX, or Phenix installation
is needed. The four table CSVs are saved alongside this notebook in `sample_bench/notebooks/exp/`.

In [1]:
from pathlib import Path
from math import comb, isfinite
import hashlib
import re

import pandas as pd
from IPython.display import display

# Works from the repository root or any notebook directory inside it.
# Set PROJECT_ROOT explicitly if launching Jupyter from elsewhere.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "sample_bench/data/analysis").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the xray repository directory.")
ANALYSIS_DIR = PROJECT_ROOT / "sample_bench/data/analysis"
OUTPUT_DIR = PROJECT_ROOT / "sample_bench/notebooks/exp"
MANUSCRIPT_PATH = PROJECT_ROOT.parent / "multi_xray_ms/main.tex"  # optional cross-check

# Metadata and deposited-PDB reference values transcribed from manuscript Tables 5–6.
# These PDB values used distinct withheld reflections, as described in the manuscript.
DATASETS = pd.DataFrame([
    ("7mhf", "100 K",  "1.55", 0.224, 0.182),
    ("7mhg", "240 K",  "1.53", 0.205, 0.160),
    ("7mhh", "277 K",  "2.19", 0.253, 0.199),
    ("7mhi", "298 K",  "1.88", 0.228, 0.191),
    ("7mhj", "298 K*", "2.00", 0.240, 0.195),
    ("7mhk", "310 K",  "1.96", 0.247, 0.198),
], columns=["dataset", "temperature", "resolution_A", "pdb_r_free", "pdb_r_work"])
CIF_IDS = DATASETS["dataset"].tolist()
ROW_ORDER = [(1, 1), (1, 2), (2, 1), (2, 2), (2, 3), (2, 4)]

# Explicit sources preserve the manuscript's experiments. Do not glob all summaries:
# alternate runs and *_copy directories would change the original table entries.
SOURCES = [
    ("before_phenix", "Matthew", "288_1_state_1_cond_ref", [(1, 1)]),
    ("before_phenix", "Matthew", "283_2_cond_ref", [(2, 1), (2, 2)]),
    ("before_phenix", "Matthew", "285_2_state_3_cond_ref", [(2, 3)]),
    ("phenix_summary", "Matthew", "288_1_state_1_cond_ref_phenix_ref", [(1, 1)]),
    ("phenix_summary", "Matthew", "283_2_cond_ref_phenix_ref", [(2, 1), (2, 2)]),
    ("phenix_summary", "Matthew", "285_2_state_3_cond_ref_phenix_ref", [(2, 3)]),
    ("phenix_summary", "Vipul", "vipul_1s2c_phenix_ref", [(1, 2)]),
    ("phenix_summary", "Vipul", "vipul_2s4c_phenix_ref", [(2, 4)]),
]

## Read summaries and retain provenance

Pre-Phenix rows can contain metrics for several datasets; refined rows usually
contain metrics for one. Each populated R-free/R-work pair becomes a candidate.
CSV index labels are retained for reference, but physical row numbers identify rows
uniquely because some source index labels repeat. Candidate counts are summary
records, not independent trajectories or unique structures.

In [2]:
records, source_records = [], []
for stage, contributor, experiment, expected_cases in SOURCES:
    path = ANALYSIS_DIR / experiment / "summary.csv"
    frame = pd.read_csv(path, index_col=0)
    required = {"N", "J", "w_xray", "pdb"}
    if not required.issubset(frame.columns) or frame.empty:
        raise ValueError(f"Missing required columns or empty summary: {path}")
    if frame[["N", "J"]].isna().any().any() or not (
        frame[["N", "J"]] == frame[["N", "J"]].astype(int)
    ).all().all():
        raise ValueError(f"N and J must be populated integers: {path}")
    actual_cases = set(zip(frame["N"].astype(int), frame["J"].astype(int)))
    if actual_cases != set(expected_cases) or not frame["w_xray"].eq(1).all():
        raise ValueError(f"Unexpected representation or w_xray in {path}")
    missing_refined_paths = 0
    for row_number, (csv_index, row) in enumerate(frame.iterrows()):
        n, j = int(row["N"]), int(row["J"])
        conditions = tuple(Path(str(row[f"cif_{k}"])).stem for k in range(j))
        if len(set(conditions)) != j or not set(conditions).issubset(CIF_IDS):
            raise ValueError(f"Invalid conditions in {experiment}, row {row_number}")
        if pd.isna(row["pdb"]) or not str(row["pdb"]).strip():
            raise ValueError(f"Missing model provenance in {experiment}, row {row_number}")
        pdb_path = str(row["pdb"])
        uncertain = stage == "phenix_summary" and "_phenix_ref" not in pdb_path
        missing_refined_paths += int(uncertain)
        populated = 0
        for dataset in CIF_IDS:
            r_free = row.get(f"r_free_{dataset}", float("nan"))
            r_work = row.get(f"r_work_{dataset}", float("nan"))
            if pd.isna(r_free) and pd.isna(r_work):
                continue
            if dataset not in conditions or not all(
                pd.notna(v) and isfinite(float(v)) and 0 <= float(v) <= 1
                for v in [r_free, r_work]
            ):
                raise ValueError(f"Invalid/unpaired metrics: {experiment}, row {row_number}, {dataset}")
            populated += 1
            records.append({
                "stage": stage, "contributor": contributor, "experiment": experiment,
                "N": n, "J": j, "dataset": dataset,
                "r_free": float(r_free), "r_work": float(r_work),
                "conditions": ", ".join(sorted(conditions)), "pdb": pdb_path,
                "source_csv": str(path.relative_to(PROJECT_ROOT)),
                "source_row": row_number, "source_csv_line": row_number + 2,
                "source_csv_index": str(csv_index), "w_xray": float(row["w_xray"]),
                "refinement_unverified": uncertain,
            })
        if populated == 0:
            raise ValueError(f"No metrics in {experiment}, row {row_number}")
    source_records.append({
        "stage": stage, "contributor": contributor, "experiment": experiment,
        "source_csv": str(path.relative_to(PROJECT_ROOT)), "rows": len(frame),
        "rows_without_phenix_path": missing_refined_paths,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    })

candidates = pd.DataFrame(records)
source_manifest = pd.DataFrame(source_records)
display(source_manifest.drop(columns=["source_csv", "sha256"]))
if candidates["refinement_unverified"].any():
    print("† Refinement status is unverified for rows without a Phenix model path; see source_manifest.")

,stage,contributor,experiment,rows,rows_without_phenix_path
0,before_phenix,Matthew,288_1_state_1_cond_ref,6,0
1,before_phenix,Matthew,283_2_cond_ref,36,0
2,before_phenix,Matthew,285_2_state_3_cond_ref,60,0
3,phenix_summary,Matthew,288_1_state_1_cond_ref_phenix_ref,6,0
4,phenix_summary,Matthew,283_2_cond_ref_phenix_ref,36,0
5,phenix_summary,Matthew,285_2_state_3_cond_ref_phenix_ref,60,0
6,phenix_summary,Vipul,vipul_1s2c_phenix_ref,30,30
7,phenix_summary,Vipul,vipul_2s4c_phenix_ref,60,0


† Refinement status is unverified for rows without a Phenix model path; see source_manifest.


In [3]:
# Verify all expected condition combinations and all six target datasets are present.
coverage_records = []
for (stage, n, j), group in candidates.groupby(["stage", "N", "J"], sort=False):
    per_dataset_combinations = group.groupby("dataset")["conditions"].nunique().reindex(CIF_IDS)
    expected_per_dataset = comb(len(CIF_IDS) - 1, j - 1)
    complete = per_dataset_combinations.eq(expected_per_dataset).all()
    coverage_records.append({
        "stage": stage, "States": n, "Conditions": j,
        "target_datasets": group["dataset"].nunique(),
        "condition_combinations": group["conditions"].nunique(),
        "expected_combinations": comb(len(CIF_IDS), j),
        "candidate_records": len(group), "complete": complete,
    })
coverage = pd.DataFrame(coverage_records)
if not coverage["complete"].all():
    raise ValueError("Incomplete condition coverage; inspect coverage before reporting minima.")
display(coverage)

# Sort first so exact R-free ties resolve reproducibly by source file and physical row.
group_keys = ["stage", "N", "J", "dataset"]
ranked = candidates.sort_values(
    group_keys + ["r_free", "source_csv", "source_row"], kind="stable"
)
selected = ranked.drop_duplicates(group_keys, keep="first").copy()
selected = selected.merge(
    candidates.groupby(group_keys).size().rename("candidate_records"),
    on=group_keys, validate="one_to_one",
)
assert not selected.duplicated(group_keys).any()
assert len(selected) == len(coverage) * len(CIF_IDS)

,stage,States,Conditions,target_datasets,condition_combinations,expected_combinations,candidate_records,complete
0,before_phenix,1,1,6,6,6,6,True
1,before_phenix,2,2,6,15,15,60,True
2,before_phenix,2,1,6,6,6,6,True
3,before_phenix,2,3,6,20,20,180,True
4,phenix_summary,1,1,6,6,6,6,True
5,phenix_summary,2,2,6,15,15,30,True
6,phenix_summary,2,1,6,6,6,6,True
7,phenix_summary,2,3,6,20,20,60,True
8,phenix_summary,1,2,6,15,15,30,True
9,phenix_summary,2,4,6,15,15,60,True


## Combined tables

The tables corresponding to manuscript Tables 5–6 include the deposited PDB reference
row and both of Vipul's additions. † marks supplied values whose refinement status
needs confirmation. The `*` on 298 K is preserved from the manuscript.

The pre-Phenix tables corresponding to Tables 3–4 follow below. `NaN` means that no
summary for that representation and stage is configured; Vipul's supplied values are
not copied into both stages.

In [4]:
LABELS = {
    row.dataset: f"{row.temperature} / {row.resolution_A} Å / {row.dataset}"
    for row in DATASETS.itertuples()
}

def build_table(stage, metric):
    rows = []
    if stage == "phenix_summary":
        rows.append({"Method": "PDB", "States": 1, "Cond.": 1,
                     **dict(zip(CIF_IDS, DATASETS[f"pdb_{metric}"]))})
    subset = selected[selected["stage"].eq(stage)]
    for n, j in ROW_ORDER:
        models = subset[subset["N"].eq(n) & subset["J"].eq(j)]
        method = "MSMC†" if models["refinement_unverified"].any() else "MSMC"
        values = models.set_index("dataset")[metric].reindex(CIF_IDS)
        rows.append({"Method": method, "States": n, "Cond.": j, **values.to_dict()})
    return pd.DataFrame(rows).set_index(["Method", "States", "Cond."])[CIF_IDS]

tables = {
    (stage, metric): build_table(stage, metric)
    for stage in ["phenix_summary", "before_phenix"]
    for metric in ["r_free", "r_work"]
}


# Plain DataFrames with ordinary columns; rounding is only for display/export.
r_free_df = tables["phenix_summary", "r_free"].rename(columns=LABELS).reset_index()
r_work_df = tables["phenix_summary", "r_work"].rename(columns=LABELS).reset_index()

with pd.option_context("display.max_columns", None):
    print("R-free — Phenix summaries (as supplied)")
    display(r_free_df.round(3))
    print("R-work — same models selected by minimum R-free")
    display(r_work_df.round(3))


R-free — Phenix summaries (as supplied)


,Method,States,Cond.,100 K / 1.55 Å / 7mhf,240 K / 1.53 Å / 7mhg,277 K / 2.19 Å / 7mhh,298 K / 1.88 Å / 7mhi,298 K* / 2.00 Å / 7mhj,310 K / 1.96 Å / 7mhk
0,PDB,1,1,0.224,0.205,0.253,0.228,0.240,0.247
1,MSMC,1,1,0.228,0.231,0.241,0.234,0.249,0.254
2,MSMC†,1,2,0.345,0.350,0.358,0.339,0.351,0.350
3,MSMC,2,1,0.223,0.226,0.251,0.235,0.249,0.258
4,MSMC,2,2,0.221,0.223,0.243,0.227,0.239,0.247
5,MSMC,2,3,0.227,0.228,0.237,0.230,0.239,0.249
6,MSMC,2,4,0.224,0.228,0.242,0.229,0.236,0.251


R-work — same models selected by minimum R-free


,Method,States,Cond.,100 K / 1.55 Å / 7mhf,240 K / 1.53 Å / 7mhg,277 K / 2.19 Å / 7mhh,298 K / 1.88 Å / 7mhi,298 K* / 2.00 Å / 7mhj,310 K / 1.96 Å / 7mhk
0,PDB,1,1,0.182,0.160,0.199,0.191,0.195,0.198
1,MSMC,1,1,0.198,0.194,0.187,0.190,0.197,0.204
2,MSMC†,1,2,0.304,0.301,0.236,0.287,0.244,0.278
3,MSMC,2,1,0.187,0.180,0.176,0.176,0.180,0.192
4,MSMC,2,2,0.185,0.182,0.176,0.176,0.183,0.187
5,MSMC,2,3,0.190,0.188,0.173,0.184,0.177,0.191
6,MSMC,2,4,0.187,0.184,0.176,0.175,0.176,0.187


In [5]:
before_phenix_r_free_df = tables["before_phenix", "r_free"].rename(columns=LABELS).reset_index()
before_phenix_r_work_df = tables["before_phenix", "r_work"].rename(columns=LABELS).reset_index()

with pd.option_context("display.max_columns", None):
    print("R-free — before Phenix refinement")
    display(before_phenix_r_free_df.round(3))
    print("R-work — before Phenix refinement, same selected models")
    display(before_phenix_r_work_df.round(3))


R-free — before Phenix refinement


,Method,States,Cond.,100 K / 1.55 Å / 7mhf,240 K / 1.53 Å / 7mhg,277 K / 2.19 Å / 7mhh,298 K / 1.88 Å / 7mhi,298 K* / 2.00 Å / 7mhj,310 K / 1.96 Å / 7mhk
0,MSMC,1,1,0.323,0.316,0.350,0.335,0.351,0.349
1,MSMC,1,2,NaN,NaN,NaN,NaN,NaN,NaN
2,MSMC,2,1,0.304,0.295,0.342,0.314,0.335,0.337
3,MSMC,2,2,0.305,0.299,0.328,0.304,0.315,0.315
4,MSMC,2,3,0.278,0.289,0.296,0.282,0.287,0.283
5,MSMC,2,4,NaN,NaN,NaN,NaN,NaN,NaN


R-work — before Phenix refinement, same selected models


,Method,States,Cond.,100 K / 1.55 Å / 7mhf,240 K / 1.53 Å / 7mhg,277 K / 2.19 Å / 7mhh,298 K / 1.88 Å / 7mhi,298 K* / 2.00 Å / 7mhj,310 K / 1.96 Å / 7mhk
0,MSMC,1,1,0.276,0.271,0.216,0.230,0.227,0.230
1,MSMC,1,2,NaN,NaN,NaN,NaN,NaN,NaN
2,MSMC,2,1,0.253,0.245,0.199,0.198,0.210,0.215
3,MSMC,2,2,0.265,0.239,0.209,0.227,0.231,0.232
4,MSMC,2,3,0.240,0.237,0.207,0.209,0.208,0.209
5,MSMC,2,4,NaN,NaN,NaN,NaN,NaN,NaN


## Selected model details

The full `selected` dataframe records both metrics, contributor, condition combination,
model path, source CSV, and physical source row for every result. The display below
shows the selected Phenix-summary results; `selected` includes both stages.

In [6]:
display(selected.loc[selected["stage"].eq("phenix_summary"), [
    "N", "J", "dataset", "r_free", "r_work", "contributor", "conditions",
    "experiment", "source_row", "refinement_unverified",
]].reset_index(drop=True))

,N,J,dataset,r_free,r_work,contributor,conditions,experiment,source_row,refinement_unverified
0,1,1,7mhf,0.228041,0.197568,Matthew,7mhf,288_1_state_1_cond_ref_phenix_ref,0,False
1,1,1,7mhg,0.230674,0.193898,Matthew,7mhg,288_1_state_1_cond_ref_phenix_ref,1,False
2,1,1,7mhh,0.240618,0.187448,Matthew,7mhh,288_1_state_1_cond_ref_phenix_ref,2,False
3,1,1,7mhi,0.234046,0.190326,Matthew,7mhi,288_1_state_1_cond_ref_phenix_ref,3,False
4,1,1,7mhj,0.248559,0.197070,Matthew,7mhj,288_1_state_1_cond_ref_phenix_ref,4,False
5,1,1,7mhk,0.253873,0.203895,Matthew,7mhk,288_1_state_1_cond_ref_phenix_ref,5,False
6,1,2,7mhf,0.345036,0.304106,Vipul,"7mhf, 7mhg",vipul_1s2c_phenix_ref,0,True
7,1,2,7mhg,0.349689,0.300621,Vipul,"7mhf, 7mhg",vipul_1s2c_phenix_ref,1,True
8,1,2,7mhh,0.357890,0.236255,Vipul,"7mhh, 7mhi",vipul_1s2c_phenix_ref,18,True
9,1,2,7mhi,0.339463,0.287293,Vipul,"7mhg, 7mhi",vipul_1s2c_phenix_ref,13,True


## Cross-check against the existing manuscript

If the manuscript is present beside the repository, compare its existing MSMC rows
to the regenerated values at the manuscript's three-decimal precision. This is an
optional cross-check; the tables themselves depend only on the configured CSV files.

In [7]:
manuscript_check = pd.DataFrame()
if MANUSCRIPT_PATH.exists():
    manuscript_text = MANUSCRIPT_PATH.read_text()
    comparisons = []
    for table_number, stage, metric in [
        (3, "before_phenix", "r_free"), (4, "before_phenix", "r_work"),
        (5, "phenix_summary", "r_free"), (6, "phenix_summary", "r_work"),
    ]:
        prefix = manuscript_text.split(r"\label{tab:table_" + str(table_number) + "}", 1)[0]
        block = prefix[prefix.rfind(r"\begin{tabular}"):]
        paper_rows = re.findall(r"MSMC\s*&\s*(\d+)\s*&\s*(\d+)\s*&([^\n]+)", block)
        if not paper_rows:
            raise ValueError(f"Cannot locate MSMC rows in manuscript Table {table_number}")
        for n_text, j_text, values_text in paper_rows:
            n, j = int(n_text), int(j_text)
            paper_values = [float(v) for v in re.findall(r"\d+\.\d+", values_text)]
            if len(paper_values) != len(CIF_IDS):
                raise ValueError(f"Unexpected manuscript columns in Table {table_number}")
            for dataset, paper_value in zip(CIF_IDS, paper_values):
                match = selected.loc[
                    selected["stage"].eq(stage) & selected["N"].eq(n) &
                    selected["J"].eq(j) & selected["dataset"].eq(dataset), metric
                ]
                actual = float(match.iloc[0]) if len(match) == 1 else float("nan")
                comparisons.append({
                    "manuscript_table": table_number, "N": n, "J": j,
                    "dataset": dataset, "metric": metric,
                    "manuscript_value": paper_value, "regenerated_value": actual,
                    "matches_3dp": f"{actual:.3f}" == f"{paper_value:.3f}",
                })
    manuscript_check = pd.DataFrame(comparisons)
    display(manuscript_check.groupby("manuscript_table")["matches_3dp"].agg(["count", "all"]))
    if not manuscript_check["matches_3dp"].all():
        display(manuscript_check.loc[~manuscript_check["matches_3dp"]])
        print("Some values differ from the current manuscript; inspect these rows before copying tables.")
else:
    print(f"Optional manuscript cross-check skipped: {MANUSCRIPT_PATH}")

,count,all
manuscript_table,,
3,24,True
4,24,True
5,24,True
6,24,True


## Save the tables as CSVs

Save the four displayed DataFrames alongside this notebook in
`sample_bench/notebooks/exp/`. Values are written to three decimals; missing
values are empty fields. † continues to mark Vipul's unverified 1-state/2-condition
refinement status. The complete model provenance remains available in `selected`.


In [8]:
csv_tables = {
    "phenix_summary_r_free.csv": r_free_df,
    "phenix_summary_r_work.csv": r_work_df,
    "before_phenix_r_free.csv": before_phenix_r_free_df,
    "before_phenix_r_work.csv": before_phenix_r_work_df,
}
for filename, frame in csv_tables.items():
    csv_path = OUTPUT_DIR / filename
    frame.to_csv(csv_path, index=False, float_format="%.3f", na_rep="")
    print(f"Saved {csv_path}")


Saved /Users/matthew/Documents/xray/sample_bench/notebooks/exp/phenix_summary_r_free.csv
Saved /Users/matthew/Documents/xray/sample_bench/notebooks/exp/phenix_summary_r_work.csv
Saved /Users/matthew/Documents/xray/sample_bench/notebooks/exp/before_phenix_r_free.csv
Saved /Users/matthew/Documents/xray/sample_bench/notebooks/exp/before_phenix_r_work.csv
